[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E6%94%BF%E5%BA%9C%E8%B3%87%E6%96%99%E7%9A%84%E4%B8%83%E7%A8%AE%E9%9B%A3%E6%90%9E%E6%A0%BC%E5%BC%8F.ipynb)

# 📥 政府資料的七種難搞格式：用 pandas 一次搞定

> 🎯 **情境**：你接下市政府的資料整理工作，第一天就收到一個資料夾：Big5 編碼還用民國年的 CSV、數字有逗號的 CSV、一層包一層的 JSON、一年一張工作表的 Excel、一個 SQLite 資料庫，還有一個大到 Excel 打開會卡住的檔案。這些都是**政府開放資料的真實樣貌**。今天的任務是：**每一種都用 pandas 一兩行讀進來，最後整理成一份 Excel 交差。**

| 素材檔案 | 內容 |
| --- | --- |
| `臺北市各區資源回收量.csv` | 臺北市各區清潔隊每月資源回收量（**Big5 編碼**，年度是**民國年**） |
| `台電各縣市用電統計.csv` | 台電各縣市每月售電量（數字有**千分位逗號**、最後有空白列和多餘的空欄） |
| `全國電影票房_2016至今.json` | 文化部全國電影票房（**巢狀 JSON**） |
| `臺北捷運各站分時進出量_202603.csv` | 臺北捷運 2026 年 3 月各站每小時進出站人次（約 8 萬列，練習**分批讀取**） |
| `台電縣市年度用電.xlsx` | 整理自台電縣市用電：一年一張工作表、**第 1 列是報表標題** |
| `空氣品質.db` | 整理自環境部日 AQI：一年份、80 個測站的 **SQLite 資料庫** |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [臺北市各區清潔隊資源回收量](https://data.gov.tw/dataset/121705) | 臺北市政府環境保護局 | 2026-10-24 |
| [台灣電力公司各縣市住宅、服務業及機關用電統計資料](https://data.gov.tw/dataset/29935) | 台灣電力股份有限公司 | 2026-10-24 |
| [全國電影票房統計數據](https://data.gov.tw/dataset/94224) | 文化部 | 2026-10-24 |
| [日空氣品質指標(AQI)](https://data.gov.tw/dataset/40507) | 環境部 | 2026-10-24 |
| [臺北捷運各站分時進出量統計](https://data.gov.tw/dataset/128506) | 臺北大眾捷運股份有限公司 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | Big5 編碼＋民國年——臺北市哪一區最會做資源回收？ |
| ⬜ 任務 2 | 千分位逗號＋空欄＋空白列——台電縣市用電 |
| ⬜ 任務 3 | 巢狀 JSON——用 `json_normalize` 攤平 |
| ⬜ 任務 4 | 一年一張工作表的 Excel |
| ⬜ 任務 5 | SQLite 資料庫——用 SQL 問問題 |
| ⬜ 任務 6 | 大檔分批讀——捷運哪一站最多人進站？ |
| ⬜ 任務 7 | 整理好了，輸出成一份 Excel 交給主管 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '臺北市各區資源回收量.csv',
    '台電各縣市用電統計.csv',
    '全國電影票房_2016至今.json',
    '臺北捷運各站分時進出量_202603.csv',
    '台電縣市年度用電.xlsx',
    '空氣品質.db',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['臺北市各區資源回收量.csv', '台電各縣市用電統計.csv', '全國電影票房_2016至今.json', '臺北捷運各站分時進出量_202603.csv', '台電縣市年度用電.xlsx', '空氣品質.db']


## 任務 1：Big5 編碼＋民國年——臺北市哪一區最會做資源回收？

**白話說明**：`encoding='cp950'` 解決亂碼；`年度` 是民國年，**加 1911** 就是西元年。

In [2]:
import pandas as pd

recycle = pd.read_csv('臺北市各區資源回收量.csv', encoding='cp950')   # Big5 檔案要指定 cp950
recycle.columns = ['年度', '月', '行政區代碼', '區隊', '回收量(噸)']       # 原始欄名有全形括號，改成好打的名字
recycle['西元年'] = recycle['年度'] + 1911                                  # 民國 → 西元
display(recycle.head())

y2025 = recycle[recycle['西元年'] == 2025]
y2025.groupby('區隊')['回收量(噸)'].sum().sort_values(ascending=False).round(0)

,年度,月,行政區代碼,區隊,回收量(噸),西元年
0,111,1,63000120.0,北投區隊,385.99,2022
1,111,1,63000010.0,松山區隊,487.38,2022
2,111,1,63000040.0,中山區隊,669.79,2022
3,111,1,63000090.0,南港區隊,243.58,2022
4,111,1,63000110.0,士林區隊,567.99,2022


區隊
大安區隊    7927.0
中山區隊    6915.0
文山區隊    6545.0
內湖區隊    5927.0
士林區隊    5547.0
信義區隊    4802.0
松山區隊    4481.0
北投區隊    4097.0
萬華區隊    3435.0
中正區隊    3142.0
南港區隊    2176.0
大同區隊    1610.0
其他       794.0
Name: 回收量(噸), dtype: float64

> 💡 **看懂結果**：2025 年回收量最多的是**大安區**（將近 8,000 噸），最少的是大同區。不過大安區人口本來就多——**要比「誰最會回收」，應該比「每個人」回收多少**。這需要把人口資料合併進來，第 10 章會教。

## 任務 2：千分位逗號＋空欄＋空白列——台電縣市用電

**白話說明**：
- `thousands=','`：把 `"636,471,432"` 讀成數字。
- 檔案最右邊有兩個沒有名字的空欄（`Unnamed: 5`、`Unnamed: 6`），最後還有一列全空白——用 `usecols` 只讀需要的欄，用 `dropna` 去掉空白列。

In [3]:
raw = pd.read_csv('台電各縣市用電統計.csv')
print('不加任何參數讀進來：')
print(raw.dtypes)                                                   # 售電量是 str（文字）！因為有逗號
print('最後兩列：'); display(raw.tail(2))

不加任何參數讀進來：
年月            float64
縣市                str
用電性質              str
售電量(度)            str
用電佔比(%)       float64
Unnamed: 5    float64
Unnamed: 6    float64
dtype: object
最後兩列：


,年月,縣市,用電性質,售電量(度),用電佔比(%),Unnamed: 5,Unnamed: 6
11263,10501.0,連江縣,工業,437623,8.88,NaN,NaN
11264,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
power = pd.read_csv('台電各縣市用電統計.csv', thousands=',', usecols=range(5))   # 只讀前 5 欄
power = power.dropna(subset=['年月'])                                             # 去掉空白列
power['年月'] = power['年月'].astype(int)
print(power.dtypes)
power.head()

年月           int64
縣市             str
用電性質           str
售電量(度)     float64
用電佔比(%)    float64
dtype: object


,年月,縣市,用電性質,售電量(度),用電佔比(%)
0,11508,新北市,住宅,997538474.0,46.22
1,11508,新北市,服務業,636471432.0,29.49
2,11508,新北市,農林漁牧,2134600.0,0.10
3,11508,新北市,工業,522024663.0,24.19
4,11508,台北市,住宅,596742074.0,38.91


> 🤖 **換你問 AI**：「我的 CSV 數字欄位有千分位逗號，最右邊還有空欄、最後一列是空白。請用 pandas 讀取並清理，讓數字欄位變成數值型態。」

## 任務 3：巢狀 JSON——用 `json_normalize` 攤平

**白話說明**：票房 JSON 的最外層是 `{'Start': ..., 'End': ..., 'List': [電影, 電影, ...]}`。`pd.json_normalize(data, record_path='List', meta=['End'])` 會把 `List` 裡的每部電影攤成一列，再把外層的 `End`（資料截止日）貼到每一列上。

In [5]:
import json

with open('全國電影票房_2016至今.json', encoding='utf-8-sig') as f:
    data = json.load(f)

movies = pd.json_normalize(data, record_path='List', meta=['End'])   # List 攤平成表格，外層的 End 也一起帶進來
print(movies.shape)
movies[['Name', 'Country', 'TotalAmounts', 'End']].head(3)

(7328, 12)


,Name,Country,TotalAmounts,End
0,《我在學校旅行中被分到了一個糟糕的小組》高校同學會－大銀幕限定青春重映,日本,122200.0,2026-10-18T00:00:00
1,YORUSHIKA LIVE「盜作」,日本,7394950.0,2026-10-18T00:00:00
2,金劫任務,美國,4353605.0,2026-10-18T00:00:00


## 任務 4：一年一張工作表的 Excel

**白話說明**：`sheet_name=None` 一次讀**所有**工作表，得到一個「工作表名稱 → 表格」的字典。這份 Excel 第 1 列是報表標題，真正的欄名在第 2 列，所以 `header=1`。最後用 `pd.concat` 上下接成一張表。

In [6]:
sheets = pd.read_excel('台電縣市年度用電.xlsx', sheet_name=None, header=1)   # 全部工作表；第 2 列才是欄名
print('工作表：', list(sheets.keys()))

yearly = pd.concat(sheets, names=['年度', '列']).reset_index(level='年度').reset_index(drop=True)
display(yearly.head())

yearly.pivot_table(index='縣市', columns='年度', values='住宅').sort_values('2025年', ascending=False).head(6)

工作表： ['2023年', '2024年', '2025年']


,年度,縣市,住宅,工業,服務業,農林漁牧
0,2023年,南投縣,9.38,11.29,6.36,1.17
1,2023年,台中市,65.15,209.39,57.19,1.66
2,2023年,台北市,54.44,6.58,90.15,0.03
3,2023年,台南市,39.71,249.70,30.21,4.45
4,2023年,台東縣,4.40,1.30,3.95,0.31


年度,2023年,2024年,2025年
縣市,,,
新北市,89.08,91.74,93.68
台中市,65.15,68.19,68.33
高雄市,59.56,62.44,61.50
桃園市,52.61,55.22,57.01
台北市,54.44,55.74,56.42
台南市,39.71,44.21,44.24


> 💡 **看懂結果**：住宅用電最多的是新北市、台中市、高雄市——人口最多的縣市。而且**大多數縣市一年比一年多**（只有高雄 2025 年略降）：家家戶戶的冷氣、電器越來越多。

## 任務 5：SQLite 資料庫——用 SQL 問問題

**白話說明**：`sqlite3.connect` 連線到資料庫檔案，`pd.read_sql(SQL 指令, 連線)` 把查詢結果直接變成 DataFrame。SQL 的 `GROUP BY` 就是 pandas 的 `groupby`。

In [7]:
import sqlite3

con = sqlite3.connect('空氣品質.db')
display(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))   # 資料庫裡有哪些資料表

worst = pd.read_sql("""
    SELECT 測站, ROUND(AVG(AQI), 1) AS 平均AQI, SUM(AQI > 100) AS 不健康天數
    FROM 日空品
    GROUP BY 測站
    ORDER BY 平均AQI DESC
    LIMIT 8
""", con)
worst

,name
0,日空品


,測站,平均AQI,不健康天數
0,斗六,75.0,69
1,小港,73.7,38
2,金門,73.6,69
3,林園,73.5,53
4,大寮,73.5,48
5,潮州,71.9,56
6,屏東,70.7,50
7,馬祖,70.4,57


## 任務 6：大檔分批讀——捷運哪一站最多人進站？

**白話說明**：遇到幾百萬列的大檔，一次讀進來可能會讓電腦當掉。`chunksize=20_000` 讓 `read_csv` **一次只給你 2 萬列**；每一批各自加總，最後再合起來。

In [8]:
total = pd.Series(dtype=float)
for chunk in pd.read_csv('臺北捷運各站分時進出量_202603.csv', chunksize=20_000):   # 每次 2 萬列
    total = total.add(chunk.groupby('車站')['進站人次'].sum(), fill_value=0)          # 這一批的結果累加上去

top = (total / 1e4).round(1).sort_values(ascending=False).rename('3月進站人次(萬)')
top.head(10)

車站
台北車站    481.4
西門      217.8
市政府     188.0
中山      161.3
忠孝復興    136.8
南京復興    125.6
BL板橋    119.5
新埔      117.9
忠孝新生    114.4
松江南京    114.2
Name: 3月進站人次(萬), dtype: float64

> 💡 **看懂結果**：2026 年 3 月，**台北車站**一個月有 481 萬人次進站，是第二名西門的兩倍多。仔細看車站名稱，有些前面多了路線代碼（例如 `BL板橋`、`G大坪林`）——這也是真實資料的「小髒」，第 8 章會處理。

## 任務 7：整理好了，輸出成一份 Excel 交給主管

**白話說明**：`pd.ExcelWriter` 可以把好幾張表寫進**同一個 Excel 的不同工作表**。存 CSV 給 Excel 開時，用 `encoding='utf-8-sig'` 才不會亂碼。

In [9]:
with pd.ExcelWriter('資料整理成果.xlsx') as writer:
    y2025.groupby('區隊')['回收量(噸)'].sum().round(0).to_excel(writer, sheet_name='2025各區回收量')
    worst.to_excel(writer, sheet_name='空品最差測站', index=False)
    top.head(20).to_excel(writer, sheet_name='捷運進站前20名')

top.to_csv('捷運進站排行.csv', encoding='utf-8-sig')                  # utf-8-sig：Excel 打開不會亂碼
print('已輸出：資料整理成果.xlsx、捷運進站排行.csv')
print(pd.read_excel('資料整理成果.xlsx', sheet_name=None).keys())

已輸出：資料整理成果.xlsx、捷運進站排行.csv
dict_keys(['2025各區回收量', '空品最差測站', '捷運進站前20名'])


<details><summary>🔄 想看最新資料？點我</summary>

環境部有提供 API，可以直接抓「這一小時」的空氣品質。把這段貼到新的一格執行（需要網路）：

```python
import pandas as pd
url = ('https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key=b7df779e-71a6-4148-8379-5afbd441d803'
       '&format=JSON&limit=100')
now = pd.read_json(url)                                    # 環境部 API：全台各測站「現在這一小時」的 AQI
now[['sitename', 'county', 'aqi', 'status', 'publishtime']].sort_values('aqi', ascending=False).head(10)
```

> ⚠️ 開放資料的欄位偶爾會改名或調整，如果最新資料跑出錯誤，正好練習看懂錯誤訊息、對照欄位名稱修正程式。

</details>

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [10]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：只讀「2024年」這一張工作表，哪個縣市的工業用電最多？

<details><summary>💡 卡住了？點我看提示</summary>

`sheet_name='2024年'` 只讀一張；別忘了 `header=1`。再用 `idxmax()` 找出「工業」最大的那一列，取出「縣市」。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
t2024 = pd.read_excel('台電縣市年度用電.xlsx', sheet_name=___, header=___)
industry_top = t2024.loc[t2024['工業'].idxmax(), '縣市']
print('2024 年工業用電最多的縣市：', industry_top)

In [ ]:
# @title ✅ 執行這格，檢查練習 1
_t = pd.read_excel('台電縣市年度用電.xlsx', sheet_name='2024年', header=1)
check('industry_top', _t.loc[_t['工業'].idxmax(), '縣市'])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
t2024 = pd.read_excel('台電縣市年度用電.xlsx', sheet_name='2024年', header=1)
industry_top = t2024.loc[t2024['工業'].idxmax(), '縣市']
print('2024 年工業用電最多的縣市：', industry_top)
```

</details>

### ✍️ 練習 2：用 SQL 查：金門有幾天 AQI 超過 100？

<details><summary>💡 卡住了？點我看提示</summary>

`WHERE 測站 = '金門' AND AQI > 100`；`COUNT(*)` 數有幾列。查詢結果是一個表格，用 `.iloc[0, 0]` 取出那個數字。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
sql = "SELECT COUNT(*) FROM 日空品 WHERE 測站 = ___ AND AQI > ___"
kinmen_days = pd.read_sql(sql, con).iloc[0, 0]
print('金門 AQI > 100 的天數：', kinmen_days)

In [ ]:
# @title ✅ 執行這格，檢查練習 2
check('kinmen_days', pd.read_sql("SELECT COUNT(*) FROM 日空品 WHERE 測站 = '金門' AND AQI > 100", con).iloc[0, 0])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
sql = "SELECT COUNT(*) FROM 日空品 WHERE 測站 = '金門' AND AQI > 100"
kinmen_days = pd.read_sql(sql, con).iloc[0, 0]
print('金門 AQI > 100 的天數：', kinmen_days)
```

</details>

### ✍️ 練習 3：臺北市 2024 年（民國 113 年）總共回收了幾噸？

<details><summary>💡 卡住了？點我看提示</summary>

可以用任務 1 做好的「西元年」欄位篩選 2024，也可以直接用民國的「年度」欄位篩選 113。最後對「回收量(噸)」加總。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
total_2024 = recycle.loc[recycle[___] == 2024, '回收量(噸)'].___()
print(f'2024 年臺北市資源回收量：{total_2024:,.0f} 噸')

In [ ]:
# @title ✅ 執行這格，檢查練習 3
check('total_2024', recycle.loc[recycle['年度'] == 113, '回收量(噸)'].sum())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
total_2024 = recycle.loc[recycle['西元年'] == 2024, '回收量(噸)'].sum()
print(f'2024 年臺北市資源回收量：{total_2024:,.0f} 噸')
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：Big5 編碼＋民國年——臺北市哪一區最會做資源回收？
- ✅ 任務 2：千分位逗號＋空欄＋空白列——台電縣市用電
- ✅ 任務 3：巢狀 JSON——用 `json_normalize` 攤平
- ✅ 任務 4：一年一張工作表的 Excel
- ✅ 任務 5：SQLite 資料庫——用 SQL 問問題
- ✅ 任務 6：大檔分批讀——捷運哪一站最多人進站？
- ✅ 任務 7：整理好了，輸出成一份 Excel 交給主管

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。